# Real-world Data Wrangling

This completed demo follows the same section order and issue-cell layout as Data_Wrangling_Project_Starter.ipynb.

It uses two small synthetic US public-library datasets. The demo intentionally contains exactly two data quality issues and two data tidiness issues. Additional pivot and melt examples appear only in the appendix so they do not change the four assessed issues.

In [ ]:
# No additional package installation is required for this local demo.
# In the Udacity workspace, install matplotlib only if it is not already available.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

DATA_DIR = Path(".")
programs_path = DATA_DIR / "library_program_registrations_dirty.csv"
branches_path = DATA_DIR / "library_branch_lookup.csv"

**Note:** Keep the notebook and both CSV files in the same directory so the relative paths work.

## 1. Gather data

In this section, load the two datasets used throughout the assessment, cleaning, storage, and analysis steps.

### **1.1.** Problem Statement

Public libraries offer programs across many subjects, but registration data must be reliable and analysis-ready before program outcomes can be compared. This demo asks which program types show the strongest score gains and which learner interests appear most often.

The two datasets are intentionally small so every assessment and cleaning decision can be demonstrated live. The separate data-gathering notebook contains the ZIP, JSON, API, and Kaggle examples; this project-flow notebook begins with the two gathered CSV files.

### **1.2.** Gather at least two datasets using two different data gathering methods

For this demo, both gathered files are already stored locally. The presentation can explain their original gathering methods, while the notebook focuses on loading and wrangling the resulting data.

#### **Dataset 1**

Type: CSV file

Method: A manually downloaded registration extract.

Dataset variables:

- registration_id identifies each registration record.
- attendance_hours measures session attendance.
- interests contains the learner interests recorded during registration.
- pre_score and post_score support learning-gain analysis.

In [ ]:
# Load the registration dataset.
programs_raw = pd.read_csv(programs_path, parse_dates=["session_date"])
programs_raw.head()

#### Dataset 2

Type: CSV file

Method: A programmatically retrieved branch lookup export.

Dataset variables:

- branch_id is the join key.
- branch_name and branch_city identify the library branch.
- region supports geographic comparison.

In [ ]:
# Load the branch lookup dataset.
branches_raw = pd.read_csv(branches_path)
branches_raw.head()

Optional data storing step: The raw CSV files remain unchanged in the local data store so the original and cleaned versions are kept separately.

In [ ]:
{
    "programs_file_exists": programs_path.exists(),
    "branches_file_exists": branches_path.exists(),
    "program_rows": len(programs_raw),
    "branch_rows": len(branches_raw),
}

## 2. Assess data

Assess exactly two data quality issues and two data tidiness issues. Each issue is inspected visually and programmatically before a cleaning strategy is selected.

### Quality Issue 1: Attendance contains missing markers and an extreme value

In [ ]:
# Visual inspection: examine the raw attendance values.
programs_raw[[
    "registration_id", "learner_name", "program_type", "attendance_hours"
]].to_string(index=False)

In [ ]:
# Programmatic inspection: discover markers before defining a missing-marker list.
attendance_raw_text = pd.read_csv(
    programs_path,
    dtype=str,
    keep_default_na=False,
)["attendance_hours"].str.strip()

attendance_as_number = pd.to_numeric(attendance_raw_text, errors="coerce")
candidate_missing_markers = (
    attendance_raw_text[attendance_as_number.isna()]
    .value_counts(dropna=False)
    .rename_axis("candidate_missing_marker")
    .reset_index(name="count")
)

attendance_median = attendance_as_number.median()
attendance_mad = (attendance_as_number - attendance_median).abs().median()
assessment_upper_bound = attendance_median + 10 * attendance_mad
extreme_attendance = programs_raw[
    pd.to_numeric(programs_raw["attendance_hours"], errors="coerce")
    > assessment_upper_bound
][["registration_id", "learner_name", "attendance_hours"]]

print("Candidate missing markers:")
print(candidate_missing_markers.to_string(index=False))
print()
print("Robust upper bound:", assessment_upper_bound)
print("Extreme attendance values:")
print(extreme_attendance.to_string(index=False))

Issue and justification: The assessment discovers TBD and * as nonnumeric attendance markers, rather than assuming them in advance. It also identifies 900 hours as an extreme value using a median absolute deviation screening rule. These related problems prevent attendance_hours from being analyzed as a reliable numeric measure.

### Quality Issue 2: Duplicate registration records

In [ ]:
# Visual inspection: sort the fields that define a registration.
programs_raw[[
    "registration_id", "learner_name", "program_id", "session_date"
]].sort_values(
    ["learner_name", "program_id", "session_date"]
).to_string(index=False)

In [ ]:
# Programmatic inspection: identify every record in a duplicate group.
duplicate_key = ["learner_name", "program_id", "session_date"]
duplicate_mask = programs_raw.duplicated(subset=duplicate_key, keep=False)
programs_raw.loc[
    duplicate_mask,
    ["registration_id", *duplicate_key],
]

Issue and justification: R010 and R011 describe the same learner, program, and session date. Counting both would overstate registrations and duplicate that learner's contribution to summaries.

### Tidiness Issue 1: Multiple interests are stored in one cell

In [ ]:
# Visual inspection: view the semicolon-separated interests.
programs_raw[["registration_id", "learner_name", "interests"]].head(10)

In [ ]:
# Programmatic inspection: count how many interests are stored per row.
interest_counts_per_row = programs_raw["interests"].str.split(";").str.len()
interest_counts_per_row.value_counts().sort_index()

Issue and justification: Each interests cell contains two values separated by a semicolon. Interest-level analysis requires one interest per row, so the column should be split and exploded.

### Tidiness Issue 2: Branch attributes are stored in a separate table

In [ ]:
# Visual inspection: compare the registration and branch structures.
print("Registration columns:")
print(programs_raw.columns.tolist())
print()
print("Branch columns:")
print(branches_raw.columns.tolist())
print()
print(branches_raw.head().to_string(index=False))

In [ ]:
# Programmatic inspection: verify the join key and its coverage.
branch_key_check = pd.Series({
    "unique_program_branch_ids": programs_raw["branch_id"].nunique(),
    "unique_lookup_branch_ids": branches_raw["branch_id"].nunique(),
    "unmatched_program_branch_ids": sorted(
        set(programs_raw["branch_id"]) - set(branches_raw["branch_id"])
    ),
})
branch_key_check

Issue and justification: Program rows contain only branch_id, while branch name, city, state, region, and facility attributes are stored in the lookup table. A validated many-to-one merge is required to create the analysis table.

#### Assessment Summary

- Quality Issue 1: Attendance contains missing markers and an extreme value.
- Quality Issue 2: Duplicate registration records.
- Tidiness Issue 1: Multiple interests are stored in one cell.
- Tidiness Issue 2: Branch attributes are stored in a separate table.

## 3. Clean data

Clean the four issues identified in the assessment. Each cleanup repeats the same issue label, applies a strategy, validates the result, and explains the decision.

In [ ]:
# Make copies so the raw DataFrames remain unchanged.
programs_clean = programs_raw.copy()
branches_clean = branches_raw.copy()

### **Quality Issue 1: Attendance contains missing markers and an extreme value**

In [ ]:
# Apply the cleaning strategy.
missing_markers = ["TBD", "*"]
programs_clean["attendance_hours"] = (
    programs_clean["attendance_hours"]
    .replace(missing_markers, np.nan)
    .pipe(pd.to_numeric, errors="coerce")
)

attendance_median = programs_clean["attendance_hours"].median()
attendance_mad = (
    programs_clean["attendance_hours"] - attendance_median
).abs().median()
outlier_upper_bound = attendance_median + 10 * attendance_mad

programs_clean.loc[
    programs_clean["attendance_hours"] > outlier_upper_bound,
    "attendance_hours",
] = np.nan

programs_clean["attendance_hours"] = (
    programs_clean.groupby("program_type")["attendance_hours"]
    .transform(lambda values: values.fillna(values.median()))
)

In [ ]:
# Validate the cleaning.
pd.Series({
    "missing_attendance_values": programs_clean["attendance_hours"].isna().sum(),
    "maximum_attendance_hours": programs_clean["attendance_hours"].max(),
    "attendance_dtype": str(programs_clean["attendance_hours"].dtype),
    "outlier_upper_bound": outlier_upper_bound,
})

Justification: The marker list comes directly from the assessment output. After conversion, the robust upper bound flags the 900-hour value without flagging the plausible 3-hour session. Missing and flagged values are imputed with the median for the same program type so registrations are retained.

### **Quality Issue 2: Duplicate registration records**

In [ ]:
# Apply the cleaning strategy.
duplicate_key = ["learner_name", "program_id", "session_date"]
programs_clean = programs_clean.drop_duplicates(
    subset=duplicate_key,
    keep="last",
).copy()

In [ ]:
# Validate the cleaning.
pd.Series({
    "remaining_duplicate_rows": programs_clean.duplicated(
        subset=duplicate_key,
        keep=False,
    ).sum(),
    "clean_registration_rows": len(programs_clean),
})

Justification: Keeping the last record applies a clear and reproducible rule. It leaves one registration for the duplicated learner-program-session combination.

### **Tidiness Issue 1: Multiple interests are stored in one cell**

In [ ]:
# Apply the cleaning strategy with split and explode.
interests_clean = (
    programs_clean
    .assign(interest=programs_clean["interests"].str.split(";"))
    .explode("interest")
    .drop(columns="interests")
)
interests_clean["interest"] = interests_clean["interest"].str.strip()

In [ ]:
# Validate the cleaning.
pd.Series({
    "rows_with_multiple_interests_remaining": interests_clean["interest"]
        .str.contains(";", regex=False)
        .sum(),
    "interest_rows": len(interests_clean),
    "unique_interests": interests_clean["interest"].nunique(),
})

Justification: Explode creates one row per registration-interest pair. This makes interest counts and groupings possible without parsing several values from one cell.

### **Tidiness Issue 2: Branch attributes are stored in a separate table**

In [ ]:
# Apply the cleaning strategy with a validated many-to-one merge.
merged = interests_clean.merge(
    branches_clean,
    on="branch_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

In [ ]:
# Validate the cleaning.
pd.Series({
    "matched_interest_rows": (merged["_merge"] == "both").sum(),
    "unmatched_interest_rows": (merged["_merge"] != "both").sum(),
    "merged_rows": len(merged),
})

Justification: A many-to-one validation confirms that each branch_id maps to at most one branch record. The merge indicator confirms that every exploded interest row receives branch attributes.

### **Remove unnecessary variables and combine datasets**

The merge audit column is useful for validation but is not needed in the final dataset. The original interests column was already replaced by the single-value interest column during the explode cleanup.

In [ ]:
# Finalize the combined datasets used for storage and analysis.
final_dataset = merged.drop(columns="_merge").copy()

registration_level = programs_clean.merge(
    branches_clean,
    on="branch_id",
    how="left",
    validate="many_to_one",
)

print("Registration-level shape:", registration_level.shape)
print("Interest-level shape:", final_dataset.shape)

## 4. Update your data store

Preserve the raw CSV files and save clearly named cleaned versions. The registration-level file is useful for program analysis, while the interest-level file is useful for interest analysis.

In [ ]:
# Save cleaned data without overwriting the raw files.
registration_clean_path = DATA_DIR / "library_programs_cleaned.csv"
interests_clean_path = DATA_DIR / "library_program_interests_cleaned.csv"

registration_level.to_csv(registration_clean_path, index=False)
final_dataset.to_csv(interests_clean_path, index=False)

print("Saved:", registration_clean_path)
print("Saved:", interests_clean_path)

## 5. Answer the research question

### **5.1:** Define and answer the research question

Use the cleaned data to compare learning outcomes by program type and learner interest frequency.

*Research question:* Which program types show the strongest average score gains, and which learner interests appear most frequently?

In [ ]:
# Visual 1: average score gain by program type.
registration_level["score_gain"] = (
    registration_level["post_score"] - registration_level["pre_score"]
)
program_type_summary = (
    registration_level.groupby("program_type")
    .agg(
        registrations=("registration_id", "count"),
        average_score_gain=("score_gain", "mean"),
    )
    .sort_values("average_score_gain", ascending=True)
)

if plt is None:
    print("Install matplotlib to display Visual 1.")
else:
    ax = program_type_summary["average_score_gain"].plot(
        kind="barh",
        figsize=(8, 5),
        color="#0F766E",
    )
    ax.set_title("Average Score Gain by Program Type")
    ax.set_xlabel("Average score gain")
    ax.set_ylabel("Program type")
    plt.tight_layout()
    plt.show()

program_type_summary.sort_values(
    "average_score_gain",
    ascending=False,
)

*Answer to research question:* STEM has the highest average score gain at 29.5 points and also has the largest number of registrations. Categories with only one registration should be interpreted cautiously because their averages are based on a single learner.

In [ ]:
# Visual 2: most common learner interests.
interest_summary = final_dataset["interest"].value_counts().head(10).sort_values()

if plt is None:
    print("Install matplotlib to display Visual 2.")
else:
    ax = interest_summary.plot(
        kind="barh",
        figsize=(8, 5),
        color="#2563EB",
    )
    ax.set_title("Most Common Learner Interests")
    ax.set_xlabel("Registration-interest rows")
    ax.set_ylabel("Interest")
    plt.tight_layout()
    plt.show()

interest_summary.sort_values(ascending=False)

*Answer to research question:* Coding is the most common interest with 5 registration-interest rows, followed by games with 4. Robots, storytelling, budgeting, writing, spreadsheets, and career each appear 3 times.

### **5.2:** Reflection

With more time, the analysis should use a larger real-world sample and investigate whether score gains differ after controlling for age, attendance, and program size. The gathering step should also document the original source and collection method for each dataset in more detail.

*Answer:* The four demonstrated issues are intentionally limited so learners can trace every assessment directly to one cleanup. Additional transformations are kept outside the assessed project flow.

## Appendix: Advanced reshaping with melt and pivot

This appendix is not an additional assessed issue. It demonstrates how the same cleaned score data can move from wide to long format and back again.

In [ ]:
# Convert pre_score and post_score from wide to long format.
scores_long = programs_clean.melt(
    id_vars=["registration_id", "learner_name", "program_type"],
    value_vars=["pre_score", "post_score"],
    var_name="assessment",
    value_name="score",
)
scores_long.head(8)

In [ ]:
# Pivot the long score table back to one row per registration.
scores_wide = (
    scores_long.pivot(
        index=["registration_id", "learner_name", "program_type"],
        columns="assessment",
        values="score",
    )
    .reset_index()
    .rename_axis(columns=None)
)
scores_wide.head()

In [ ]:
# Validate the melt-pivot round trip.
expected_scores = (
    programs_clean[[
        "registration_id", "learner_name", "program_type",
        "pre_score", "post_score",
    ]]
    .sort_values("registration_id")
    .reset_index(drop=True)
)
round_trip_scores = (
    scores_wide[expected_scores.columns]
    .sort_values("registration_id")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    expected_scores,
    round_trip_scores,
    check_dtype=False,
)
print("Melt-pivot round trip validated.")